# RAG Evaluation: Labeling and Results

This notebook turns `evaluation/results.jsonl` (produced by `python src/evaluate.py`) into the
**Table I** used in the paper.

**Workflow**
1. Run the labeling cell below (ipywidgets UI). For each query, read the **Baseline** (no RAG) and
   **RAG** answers and tag each as `correct`, `irrelevant`, or `hallucinated`.
2. Click **Save Labels**.
3. Run the *Table I* cell to get counts and percentages for the paper.
4. Use the per-category, retrieval-metric and error-analysis cells for the Discussion section.

Labeling rubric:
- **correct**: accurate and consistent with the source documents (a proper *I don't know* on an
  out-of-scope / withheld query is *correct*).
- **irrelevant**: does not address the question asked.
- **hallucinated**: invents information not supported by the documents (including inventing an
  answer when the information is unavailable).

In [3]:
import os, json
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import pandas as pd

_base = os.getcwd()
PROJECT_DIR = os.path.dirname(_base) if os.path.basename(_base) == "notebooks" else _base
EVAL_DIR = os.path.join(PROJECT_DIR, "evaluation")
RESULTS_PATH = os.path.join(EVAL_DIR, "results.jsonl")
LABELS_PATH = os.path.join(EVAL_DIR, "labels.json")

results = [json.loads(l) for l in open(RESULTS_PATH, encoding="utf-8") if l.strip()]
print(f"Project: {PROJECT_DIR}")
print(f"Loaded {len(results)} evaluated queries")

Project: e:\Main_pro\NLPv3
Loaded 25 evaluated queries


## 1. Label the responses
Tag both answers for every query, then click **Save Labels**. Re-running this cell resumes from saved labels.

In [4]:
state = {"i": 0}

labels = {str(r["id"]): {"baseline": None, "rag": None} for r in results}
if os.path.exists(LABELS_PATH):
    labels = json.load(open(LABELS_PATH, encoding="utf-8"))

OPTIONS = ["correct", "irrelevant", "hallucinated"]

idx_label = widgets.Label()
prompt_out = widgets.Output()
b_radio = widgets.RadioButtons(options=OPTIONS, description="Baseline:")
r_radio = widgets.RadioButtons(options=OPTIONS, description="RAG:")
prev_btn = widgets.Button(description="< Previous")
next_btn = widgets.Button(description="Next >")
save_btn = widgets.Button(description="Save Labels")
status = widgets.Label()

def _record():
    res = results[state["i"]]
    labels[str(res["id"])]["baseline"] = b_radio.value
    labels[str(res["id"])]["rag"] = r_radio.value

def render():
    res = results[state["i"]]
    idx_label.value = (f"Query {state['i']+1}/{len(results)}  |  "
                       f"category: {res['category']}  |  answer_in_kb: {res['answer_in_kb']}")
    b_radio.value = labels.get(str(res["id"]), {}).get("baseline")
    r_radio.value = labels.get(str(res["id"]), {}).get("rag")
    with prompt_out:
        clear_output(wait=True)
        print("QUESTION:", res["query"])
        print("\n--- Baseline (LLM, no retrieval) ---")
        print(res["baseline_answer"])
        print("\n--- RAG (LLM + retrieval) ---")
        print(res["rag_answer"])
        if res.get("rag_sources"):
            src = [f"{s['doc_name']}#{s['chunk_id']}" for s in res["rag_sources"]]
            print("\nRAG sources:", src)

def on_prev(b):
    _record()
    if state["i"] > 0:
        state["i"] -= 1
    render()

def on_next(b):
    _record()
    if state["i"] < len(results) - 1:
        state["i"] += 1
        render()
    else:
        status.value = "Last query. Click Save Labels."

def on_save(b):
    for r in results:
        labels.setdefault(str(r["id"]), {"baseline": None, "rag": None})
    json.dump(labels, open(LABELS_PATH, "w", encoding="utf-8"), indent=2)
    n = sum(1 for r in results if labels[str(r["id"])]["baseline"] and labels[str(r["id"])]["rag"])
    status.value = f"Saved. {n}/{len(results)} queries fully labeled -> {LABELS_PATH}"

prev_btn.on_click(on_prev)
next_btn.on_click(on_next)
save_btn.on_click(on_save)

display(widgets.VBox([idx_label, prompt_out, b_radio, r_radio,
                     widgets.HBox([prev_btn, next_btn, save_btn]), status]))
render()

## 2. Table I - Response quality with and without retrieval
Use these numbers for Table I in the paper.

In [5]:
LABELS_ORDER = ["correct", "irrelevant", "hallucinated"]

def summarize(config_key):
    counts = {l: 0 for l in LABELS_ORDER}
    unlabeled = 0
    for r in results:
        lab = labels.get(str(r["id"]), {}).get(config_key)
        if lab in counts:
            counts[lab] += 1
        else:
            unlabeled += 1
    return counts, unlabeled

base_counts, base_unl = summarize("baseline")
rag_counts, rag_unl = summarize("rag")
total = len(results)

table = pd.DataFrame({
    "Correct": [base_counts["correct"], rag_counts["correct"]],
    "Irrelevant": [base_counts["irrelevant"], rag_counts["irrelevant"]],
    "Hallucinated": [base_counts["hallucinated"], rag_counts["hallucinated"]],
    "Total": [total, total],
}, index=["LLM without RAG", "LLM with RAG"])

print(table.to_string())
print()
print(f"Unlabeled -> baseline: {base_unl}, rag: {rag_unl}")
print()
pct = (table[["Correct", "Irrelevant", "Hallucinated"]] / total * 100).round(1)
print("Percentages:")
print(pct.to_string())

                 Correct  Irrelevant  Hallucinated  Total
LLM without RAG        1           0             0     25
LLM with RAG           1           0             0     25

Unlabeled -> baseline: 24, rag: 24

Percentages:
                 Correct  Irrelevant  Hallucinated
LLM without RAG      4.0         0.0           0.0
LLM with RAG         4.0         0.0           0.0


## 3. Per-category breakdown

In [6]:
cats = sorted({r["category"] for r in results})
rows = []
for c in cats:
    row = {"category": c}
    for cfg in ("baseline", "rag"):
        cc = {l: 0 for l in LABELS_ORDER}
        t = 0
        for r in results:
            if r["category"] != c:
                continue
            t += 1
            lab = labels.get(str(r["id"]), {}).get(cfg)
            if lab in cc:
                cc[lab] += 1
        row[f"{cfg}_correct"] = cc["correct"]
        row[f"{cfg}_irrel"] = cc["irrelevant"]
        row[f"{cfg}_halluc"] = cc["hallucinated"]
        row[f"{cfg}_total"] = t
    rows.append(row)

cat_df = pd.DataFrame(rows)
print(cat_df.to_string(index=False))

    category  baseline_correct  baseline_irrel  baseline_halluc  baseline_total  rag_correct  rag_irrel  rag_halluc  rag_total
  admissions                 1               0                0               5            1          0           0          5
examinations                 0               0                0               5            0          0           0          5
        fees                 0               0                0               5            0          0           0          5
      hostel                 0               0                0               5            0          0           0          5
     library                 0               0                0               5            0          0           0          5


## 4. Retrieval quality and latency (optional metrics)
- **Retrieval hit rate**: fraction of answerable queries whose `expected_doc` appears in the RAG top-k.
- **Latency**: mean end-to-end time per configuration.

In [7]:
hits = 0
considered = 0
for r in results:
    if r.get("answer_in_kb") and r.get("expected_doc"):
        considered += 1
        docs = [s["doc_name"] for s in r.get("rag_sources", [])]
        if r["expected_doc"] in docs:
            hits += 1
if considered:
    print(f"Retrieval hit rate: {hits}/{considered} = {hits/considered*100:.1f}%")

bt = [r.get("baseline_time_s", 0) for r in results]
rt = [r.get("rag_time_s", 0) for r in results]
print(f"Avg baseline latency: {np.mean(bt):.2f}s  |  Avg RAG latency: {np.mean(rt):.2f}s")

k_values = [len(r.get("rag_sources", [])) for r in results]
print(f"Avg sources retrieved per query: {np.mean(k_values):.2f} (k={TOP_K})")

Retrieval hit rate: 20/20 = 100.0%
Avg baseline latency: 7.29s  |  Avg RAG latency: 5.85s


NameError: name 'TOP_K' is not defined

## 5. Error analysis
List the responses labelled *hallucinated* to inspect for the Discussion section.

In [8]:
for cfg in ("baseline", "rag"):
    flagged = [r for r in results if labels.get(str(r["id"]), {}).get(cfg) == "hallucinated"]
    print(f"\n===== {cfg.upper()}: {len(flagged)} hallucinated =====")
    for r in flagged:
        ans = r["baseline_answer"] if cfg == "baseline" else r["rag_answer"]
        print(f"\n[id {r['id']}] {r['query']}")
        print(ans[:400])
        print("-" * 60)


===== BASELINE: 0 hallucinated =====

===== RAG: 0 hallucinated =====
